In [0]:
eventhub_conn_str = dbutils.secrets.get(scope="crypto-secrets", key="eventhub-connection-string")
storage_account_name = "cryptolakesalman"  # <-- your actual name
print("Secrets loaded. Storage authentication handled via cluster-level Spark config.")

Secrets loaded. Storage authentication handled via cluster-level Spark config.


In [0]:
# Event Hub Kafka-compatible endpoint config
eventhub_namespace = "crypto-eventhub-salman"  # <-- your namespace name, update if different
eventhub_name = "crypto-trades"

bootstrap_servers = f"{eventhub_namespace}.servicebus.windows.net:9093"

kafka_options = {
    "kafka.bootstrap.servers": bootstrap_servers,
    "subscribe": eventhub_name,
    "kafka.sasl.mechanism": "PLAIN",
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.jaas.config": f'org.apache.kafka.common.security.plain.PlainLoginModule required username="$ConnectionString" password="{eventhub_conn_str}";',
    "startingOffsets": "latest",
    "failOnDataLoss": "false"   # <-- add this line

}

print("Kafka-compatible Event Hub config ready.")

Kafka-compatible Event Hub config ready.


In [0]:
raw_stream_df = (
    spark.readStream
    .format("kafka")
    .options(**kafka_options)
    .load()
)

print("Streaming DataFrame created. Schema:")
raw_stream_df.printSchema()

Streaming DataFrame created. Schema:
root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



In [0]:
from pyspark.sql.functions import col, from_json, current_timestamp
from pyspark.sql.types import StructType, StringType, LongType

# Define the schema matching what your Python script sends
trade_schema = StructType() \
    .add("symbol", StringType()) \
    .add("price", StringType()) \
    .add("quantity", StringType()) \
    .add("trade_time", LongType())

# Decode the Kafka 'value' column (bytes -> string -> JSON)
parsed_df = (
    raw_stream_df
    .selectExpr("CAST(value AS STRING) as json_value", "timestamp as kafka_timestamp")
    .withColumn("data", from_json(col("json_value"), trade_schema))
    .select("data.*", "kafka_timestamp")
    .withColumn("ingestion_time", current_timestamp())
)

print("Parsed DataFrame schema:")
parsed_df.printSchema()

Parsed DataFrame schema:
root
 |-- symbol: string (nullable = true)
 |-- price: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- trade_time: long (nullable = true)
 |-- kafka_timestamp: timestamp (nullable = true)
 |-- ingestion_time: timestamp (nullable = false)



In [0]:
bronze_path = f"abfss://bronze@{storage_account_name}.dfs.core.windows.net/crypto_trades"
checkpoint_path = f"abfss://bronze@{storage_account_name}.dfs.core.windows.net/_checkpoints/crypto_trades"

query = (
    parsed_df
    .writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .outputMode("append")
    .start(bronze_path)
)

print("Streaming write started! Writing live trades to bronze layer...")

In [0]:
# Peek at what's been written so far
df_check = spark.read.format("delta").load(bronze_path)
print(f"Total records written: {df_check.count()}")
df_check.orderBy(col("ingestion_time").desc()).show(10, truncate=False)

Total records written: 17851
+-------+--------------+----------+-------------+-----------------------+-----------------------+
|symbol |price         |quantity  |trade_time   |kafka_timestamp        |ingestion_time         |
+-------+--------------+----------+-------------+-----------------------+-----------------------+
|BTCUSDT|62749.33000000|0.00011000|1783229147755|2026-07-05 05:25:48.21 |2026-07-05 05:25:49.827|
|BTCUSDT|62749.33000000|0.00025000|1783229148590|2026-07-05 05:25:48.851|2026-07-05 05:25:49.827|
|BTCUSDT|62749.33000000|0.00013000|1783229146919|2026-07-05 05:25:47.631|2026-07-05 05:25:49.827|
|BTCUSDT|62749.33000000|0.00007000|1783229148590|2026-07-05 05:25:48.741|2026-07-05 05:25:49.827|
|BTCUSDT|62749.32000000|0.00350000|1783229129041|2026-07-05 05:25:29.661|2026-07-05 05:25:47.523|
|BTCUSDT|62749.32000000|0.14214000|1783229129041|2026-07-05 05:25:29.395|2026-07-05 05:25:47.523|
|BTCUSDT|62749.33000000|0.00085000|1783229128330|2026-07-05 05:25:28.552|2026-07-05 05:25

In [0]:
print(storage_account_name)



cryptolakesalman


In [0]:
# query.stop()
# print("Streaming query stopped cleanly.")

In [0]:
df_check = spark.read.format("delta").load(bronze_path)
print(f"Total records written: {df_check.count()}")

In [0]:
spark.sql("SELECT * FROM crypto_databricks_salman.crypto_silver_schema.silver_crypto_trades ORDER BY trade_timestamp DESC LIMIT 10").show(truncate=False)

+-------+--------+--------+-------------------+-----------------------+
|symbol |price   |quantity|trade_timestamp    |ingestion_time         |
+-------+--------+--------+-------------------+-----------------------+
|BTCUSDT|62520.76|9.0E-5  |2026-07-04 11:22:03|2026-07-04 11:22:51.527|
|BTCUSDT|62521.95|8.0E-5  |2026-07-04 11:22:03|2026-07-04 11:22:48.975|
|BTCUSDT|62521.99|9.0E-5  |2026-07-04 11:22:03|2026-07-04 11:22:42.395|
|BTCUSDT|62521.99|1.4E-4  |2026-07-04 11:22:03|2026-07-04 11:22:40.704|
|BTCUSDT|62521.98|8.0E-5  |2026-07-04 11:22:03|2026-07-04 11:22:47.168|
|BTCUSDT|62521.9 |0.05    |2026-07-04 11:22:03|2026-07-04 11:22:49.805|
|BTCUSDT|62518.0 |9.0E-5  |2026-07-04 11:22:03|2026-07-04 11:23:00.525|
|BTCUSDT|62521.97|1.6E-4  |2026-07-04 11:22:03|2026-07-04 11:22:47.168|
|BTCUSDT|62521.95|9.0E-5  |2026-07-04 11:22:03|2026-07-04 11:22:48.114|
|BTCUSDT|62521.99|0.03198 |2026-07-04 11:22:03|2026-07-04 11:22:44.118|
+-------+--------+--------+-------------------+-----------------

In [0]:
spark.sql("SELECT * FROM crypto_databricks_salman.crypto_silver_schema.gold_crypto_ohlc_1min ORDER BY window_start DESC LIMIT 10").show(truncate=False)


+-------+-------------------+-------------------+----------+----------+---------+-----------+------------------+-----------+------------------+
|symbol |window_start       |window_end         |open_price|high_price|low_price|close_price|total_volume      |trade_count|avg_price         |
+-------+-------------------+-------------------+----------+----------+---------+-----------+------------------+-----------+------------------+
|BTCUSDT|2026-07-04 11:22:00|2026-07-04 11:23:00|62521.99  |62521.99  |62518.0  |62520.44   |0.30728           |38         |62521.47789473685 |
|BTCUSDT|2026-07-04 11:21:00|2026-07-04 11:22:00|62514.0   |62522.0   |62504.67 |62516.0    |1.2317399999999992|259        |62513.16714285713 |
|BTCUSDT|2026-07-04 11:20:00|2026-07-04 11:21:00|62501.54  |62504.68  |62501.54 |62501.67   |1.045610000000001 |151        |62501.78072847678 |
|BTCUSDT|2026-07-04 11:19:00|2026-07-04 11:20:00|62501.54  |62501.55  |62501.54 |62501.54   |0.1912            |80         |62501.543624

In [0]:
# Check: does trade_count per window roughly match total silver rows / number of windows?
result = spark.sql("""
    SELECT 
        window_start, 
        window_end, 
        trade_count 
    FROM crypto_databricks_salman.crypto_silver_schema.gold_crypto_ohlc_1min
    ORDER BY window_start
""")
result.show(30, truncate=False)

# Sum of all trade_count across all gold rows should equal (or be very close to) silver row count
total_check = spark.sql("""
    SELECT SUM(trade_count) AS sum_of_trade_counts
    FROM crypto_databricks_salman.crypto_silver_schema.gold_crypto_ohlc_1min
""")
total_check.show()

+-------------------+-------------------+-----------+
|window_start       |window_end         |trade_count|
+-------------------+-------------------+-----------+
|2026-07-02 13:15:00|2026-07-02 13:16:00|383        |
|2026-07-02 13:16:00|2026-07-02 13:17:00|50         |
|2026-07-02 18:17:00|2026-07-02 18:18:00|329        |
|2026-07-02 18:18:00|2026-07-02 18:19:00|539        |
|2026-07-02 18:25:00|2026-07-02 18:26:00|322        |
|2026-07-02 18:26:00|2026-07-02 18:27:00|345        |
|2026-07-02 18:31:00|2026-07-02 18:32:00|492        |
|2026-07-02 18:36:00|2026-07-02 18:37:00|2          |
|2026-07-02 18:37:00|2026-07-02 18:38:00|404        |
|2026-07-02 18:38:00|2026-07-02 18:39:00|218        |
|2026-07-03 10:18:00|2026-07-03 10:19:00|122        |
|2026-07-03 10:21:00|2026-07-03 10:22:00|196        |
|2026-07-03 10:22:00|2026-07-03 10:23:00|455        |
|2026-07-03 10:23:00|2026-07-03 10:24:00|514        |
|2026-07-03 10:34:00|2026-07-03 10:35:00|254        |
|2026-07-03 10:35:00|2026-07